# Agentic research workflows on Google Cloud — Colab mirror

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanyaasharma02/ai4science-agentic-lab/blob/main/notebooks/lab.ipynb)

Same lab as the repo, as cells. Modules 01–04. Source + README: https://github.com/tanyaasharma02/ai4science-agentic-lab

**Setup:** run the next two cells, fill in your project, then run the modules top to bottom.

In [ ]:
!git clone -q https://github.com/tanyaasharma02/ai4science-agentic-lab.git /content/lab_repo 2>/dev/null || (cd /content/lab_repo && git pull -q)
%cd /content/lab_repo
!pip install -q -r requirements.txt
print('installed')

In [ ]:
# --- credentials: pick ONE ---
import os
from google.colab import auth

# Option A (recommended, uses your Cloud Credits): Vertex AI
auth.authenticate_user()
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = 'YOUR_PROJECT_ID'   # <-- edit
os.environ['GOOGLE_CLOUD_LOCATION'] = 'us-central1'      # try 'global' if a model 404s

# Option B: AI Studio key (no GCP project needed)
# os.environ['GOOGLE_API_KEY'] = 'AIza...'

# Optional: if your project lacks gemini-2.5-pro, use a flash model as the critic
# os.environ['LAB_STRONG_MODEL'] = 'gemini-2.5-flash'
!python setup/verify_env.py

## Module 01 — one agent, your code as a tool
The docstring of `run_simulation` *is* the tool spec. Watch the agent plan a sweep on its own, then delegate the curve fit to an `analyst` agent (built-in code execution lives in its own agent — Vertex won't mix it with function tools).

In [ ]:
from lab.common import run
from lab.modules.m01_single_agent_tools import experiment_agent, analyst_agent
from google.adk.agents import Agent
from google.adk.tools.agent_tool import AgentTool
from lab.common import FAST_MODEL
from lab.tools.simulation import run_simulation

run(experiment_agent, 'What is the steady-state amplitude with default parameters?')

In [ ]:
# TODO(2) solved: give the agent the analyst
agent_with_analyst = Agent(name='experiment_agent', model=FAST_MODEL, instruction=experiment_agent.instruction,
                           tools=[run_simulation, AgentTool(agent=analyst_agent)])
run(agent_with_analyst, 'Sweep drive_frequency over 0.4, 0.7, 1.0, 1.3, 1.6 with stiffness=1.0 and damping=0.1. '
                        'Where is the resonance peak and how sharp is it? Fit a curve rather than eyeballing.')

## Module 02 — vision as a tool
`analyse_image` asks Gemini a focused question about an image and returns **structured JSON**. The agent uses it to sanity-check its own plot.

In [ ]:
from lab.modules.m02_multimodal_tool import experiment_agent as mm_agent
run(mm_agent, 'Run the oscillator with damping=0.02, drive_frequency=1.0 and duration_s=20. '
              'Did it reach steady state? Check the plot, don\'t just trust the number.')

In [ ]:
from IPython.display import Image, display
import glob; display(Image(sorted(glob.glob('lab/outputs/sim_*.png'))[-1], width=600))

In [ ]:
# TODO(3): how far off is the VLM on count / size?
run(mm_agent, 'Generate a synthetic micrograph with 30 particles of mean radius 10 px, '
              'then estimate the particle count and mean radius from the image and compare to ground truth.')

## Module 03 — the mini co-scientist
`LoopAgent`: hypothesis → experiment → critic, sharing state via `output_key`. The critic ends the loop with `accept(conclusion)`. Watch for a **REVISE** round — that's the point.

In [ ]:
from lab.modules.m03_multi_agent_loop import mini_co_scientist
run(mini_co_scientist, 'How does the resonance peak amplitude of the oscillator depend on damping? '
                       'I care about damping between 0.02 and 0.5.')

## Module 04 — human-in-the-loop gate + commit
The reporter **must** call `request_human_approval` before `commit_result`. In Colab the prompt appears inline; type `y`.

In [ ]:
from lab.modules.m04_hitl_and_sharing import root_agent
run(root_agent, 'Does the resonance peak get sharper (narrower) as damping decreases? Test with damping 0.05 vs 0.3.')
!tail -n 1 lab/outputs/results.jsonl

## Module 05 — swap in your code
Open `lab/modules/m05_swap_in_your_code.py`. Three rules: typed args + docstring, small JSON return, validate inputs inside the tool. Point module 03's experiment agent at your function and you're done.

**Sharing with your lab:** see `deploy/agent_engine.py` and the README section *Sharing the agent with your lab*.